# Qwen activation cosine similarities

Each curve compares token vectors with that layer/type's mean vector.

In [ ]:
import math
from itertools import islice

import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from plotly.subplots import make_subplots
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-0.6B'  # any Hugging Face Qwen3 causal-LM checkpoint
DATASET_ID = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
N_TEXTS = 256
BATCH_SIZE = 8
MAX_LENGTH = 256
PAIRS_PER_TEXT = 256  # random within-text token pairs for each histogram
INTERLAYER_TOKENS_PER_BATCH = 32  # corresponding token positions for layer-pair plots
PCA_VECTORS_PER_TYPE = 512  # per layer/type; bounds PCA memory

dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split='train', streaming=True)
dataset = dataset.shuffle(seed=0, buffer_size=10_000)
TEXTS = [row['text'] for row in islice(dataset, N_TEXTS) if row['text'].strip()]
PCA_SAMPLES_PER_BATCH = math.ceil(PCA_VECTORS_PER_TYPE / math.ceil(len(TEXTS) / BATCH_SIZE))
torch.manual_seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()


In [ ]:
# Keep sampled within-text pairwise cosine scalars; retain only a small vector sample for PCA.
names = ('q', 'k', 'v', 'residual', 'residual_write')
layers = model.model.layers
activations = {i: {name: [] for name in names} for i in range(len(layers))}
pca_activations = {i: {name: [] for name in names} for i in range(len(layers))}
interlayer_parts = {kind: [[[] for _ in layers] for _ in layers] for kind in ('residual', 'residual_write')}
embedding_pairwise_similarities = []
embedding_to_layer0_similarities = []
captured = {}
hooks = []

def save_projection(layer_idx, name):
    def hook(module, inputs, output):
        captured[layer_idx][name] = output.detach().float().cpu()
    return hook

def save_residual(layer_idx):
    def hook(_, inputs, output):
        residual_in = inputs[0]
        residual_out = output[0] if isinstance(output, tuple) else output
        captured[layer_idx]['residual'] = residual_in.detach().float().cpu()
        captured[layer_idx]['residual_write'] = (residual_out - residual_in).detach().float().cpu()
    return hook

for layer_idx, layer in enumerate(layers):
    hooks += [
        layer.register_forward_hook(save_residual(layer_idx)),
        layer.self_attn.q_proj.register_forward_hook(save_projection(layer_idx, 'q')),
        layer.self_attn.k_proj.register_forward_hook(save_projection(layer_idx, 'k')),
        layer.self_attn.v_proj.register_forward_hook(save_projection(layer_idx, 'v')),
    ]

def sample_within_text_pairwise_cosines(vectors, valid_tokens):
    similarities = []
    for text_vectors, text_mask in zip(vectors, valid_tokens):
        text_vectors = F.normalize(text_vectors[text_mask], dim=-1)
        n_tokens = text_vectors.size(0)
        if n_tokens < 2:
            continue
        n_pairs = min(PAIRS_PER_TEXT, n_tokens * (n_tokens - 1) // 2)
        first = torch.randint(n_tokens, (n_pairs,))
        second = torch.randint(n_tokens - 1, (n_pairs,))
        second += second >= first  # never compare a token with itself
        similarities.append((text_vectors[first] * text_vectors[second]).sum(-1))
    return torch.cat(similarities)

def append_pca_sample(parts, vectors):
    remaining = PCA_VECTORS_PER_TYPE - sum(part.size(0) for part in parts)
    if remaining > 0:
        indices = torch.randperm(vectors.size(0))[:min(remaining, PCA_SAMPLES_PER_BATCH)]
        parts.append(vectors[indices])

def append_interlayer_similarities(kind, valid_tokens):
    states = torch.stack([captured[layer_idx][kind] for layer_idx in range(len(layers))])[:, valid_tokens]
    indices = torch.randperm(states.size(1))[:INTERLAYER_TOKENS_PER_BATCH]
    states = F.normalize(states[:, indices], dim=-1)
    similarities = torch.einsum('ltd,mtd->lmt', states, states)
    for left in range(len(layers)):
        for right in range(left, len(layers)):
            interlayer_parts[kind][left][right].append(similarities[left, right])

for start in range(0, len(TEXTS), BATCH_SIZE):
    batch = tokenizer(TEXTS[start:start + BATCH_SIZE], return_tensors='pt', padding=True,
                      truncation=True, max_length=MAX_LENGTH).to(device)
    captured = {i: {} for i in range(len(layers))}
    with torch.inference_mode():
        model(**batch, use_cache=False)
    valid_tokens = batch.attention_mask.bool().cpu()
    embeddings = model.get_input_embeddings()(batch.input_ids).detach().float().cpu()
    embedding_pairwise_similarities.append(sample_within_text_pairwise_cosines(embeddings, valid_tokens))
    embedding_to_layer0_similarities.append(F.cosine_similarity(
        embeddings[valid_tokens], captured[0]['residual'][valid_tokens], dim=-1
    ))
    for layer_idx, layer in captured.items():
        for name, value in layer.items():
            activations[layer_idx][name].append(sample_within_text_pairwise_cosines(value, valid_tokens))
            append_pca_sample(pca_activations[layer_idx][name], value[valid_tokens])
    append_interlayer_similarities('residual', valid_tokens)
    append_interlayer_similarities('residual_write', valid_tokens)

for hook in hooks:
    hook.remove()
embedding_pairwise_similarities = torch.cat(embedding_pairwise_similarities)
embedding_to_layer0_similarities = torch.cat(embedding_to_layer0_similarities)
print('Embedding ↔ layer-0 residual input cosine:',
      f'mean={embedding_to_layer0_similarities.mean():.6f}, min={embedding_to_layer0_similarities.min():.6f}')
for layer_idx in activations:
    for name in names:
        activations[layer_idx][name] = torch.cat(activations[layer_idx][name])
        pca_activations[layer_idx][name] = torch.cat(pca_activations[layer_idx][name])

interlayer_similarities = {
    kind: [[torch.cat(interlayer_parts[kind][min(i, j)][max(i, j)]) for j in range(len(layers))]
           for i in range(len(layers))]
    for kind in interlayer_parts
}
print('Layer-0 residual (= embeddings) similarity to residual inputs:')
for layer_idx, values in enumerate(interlayer_similarities['residual'][0]):
    print(f'  layer {layer_idx:2}: mean={values.mean():.4f}  n={values.numel()}')


In [ ]:
def n_bins(values):
    return max(20, min(200, math.ceil(math.sqrt(values.numel()))))

colors = {'embedding': '#19D3F3', 'q': '#636EFA', 'k': '#EF553B', 'v': '#00CC96',
          'residual': '#AB63FA', 'residual_write': '#FFA15A'}
names = ('q', 'k', 'v', 'residual', 'residual_write')
n_panels = len(activations) + 1
cols = min(4, n_panels)
rows = math.ceil(n_panels / cols)
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=['Input embeddings'] + [f'Layer {i}' for i in activations],
)

fig.add_trace(go.Histogram(
    x=embedding_pairwise_similarities, name='embedding', marker_color=colors['embedding'],
    opacity=0.55, nbinsx=n_bins(embedding_pairwise_similarities), legendgroup='embedding',
), row=1, col=1)
for layer_idx, layer in activations.items():
    row, col = divmod(layer_idx + 1, cols)
    layer_bins = n_bins(torch.cat([layer[name] for name in names]))
    for name in names:
        fig.add_trace(go.Histogram(
            x=layer[name], name=name.replace('_', ' '), marker_color=colors[name],
            opacity=0.55, nbinsx=layer_bins, legendgroup=name, showlegend=layer_idx == 0,
        ), row=row + 1, col=col + 1)

fig.update_layout(
    barmode='overlay', height=280 * rows, width=360 * cols,
    title='Within-text token-pair cosine similarity',
)
fig.update_xaxes(title_text='cosine similarity')
fig.update_yaxes(title_text='count')
fig.show()


In [ ]:
# Change this to inspect a different transformer layer.
PCA_LAYER = 10

pca_vectors = pca_activations[PCA_LAYER]
torch.manual_seed(0)  # torch.pca_lowrank is randomized

fig_pca = go.Figure()
for name, vectors in pca_vectors.items():
    # Q and K/V have different widths under grouped-query attention.
    _, _, components = torch.pca_lowrank(vectors - vectors.mean(0), q=2, center=False)
    coordinates = (vectors - vectors.mean(0)) @ components[:, :2]
    fig_pca.add_trace(go.Scattergl(
        x=coordinates[:, 0], y=coordinates[:, 1], mode='markers', name=name.replace('_', ' '),
        marker={'color': colors[name], 'size': 5, 'opacity': 0.45},
    ))

fig_pca.update_layout(
    title=f'Independent 2D PCA of Q/K/V and residual streams — layer {PCA_LAYER}',
    xaxis_title='PC 1', yaxis_title='PC 2', width=900, height=650,
)
fig_pca.show()


In [ ]:
COOLWARM = [[0, '#3b4cc0'], [0.5, '#dddcdc'], [1, '#b40426']]
REDS = [[0, '#dddcdc'], [0.5, '#fb6a4a'], [1, '#b40426']]
VIOLIN_BINS = 64
VIOLIN_SMOOTHING = torch.tensor([1, 4, 6, 4, 1], dtype=torch.float32).view(1, 1, -1) / 16

def add_similarity_heatmap_with_violins(fig, kind, column):
    values = interlayer_similarities[kind]
    n_layers = len(values)
    means = [[values[i][j].mean().item() for j in range(n_layers)] for i in range(n_layers)]
    fig.add_trace(go.Heatmap(
        z=means, x=list(range(n_layers)), y=list(range(n_layers)), colorscale=REDS if kind == 'residual' else COOLWARM,
        zmin=0 if kind == 'residual' else -1, zmax=1,
        colorbar={'title': 'mean cosine', 'x': 0.46 if column == 1 else 1.0},
        hovertemplate='layer %{y} ↔ layer %{x}<br>mean cosine: %{z:.3f}<extra></extra>',
    ), row=1, col=column)

    x, y = [], []
    centers = torch.linspace(-1 + 1 / VIOLIN_BINS, 1 - 1 / VIOLIN_BINS, VIOLIN_BINS)
    for left in range(n_layers):
        for right in range(n_layers):
            density = torch.histc(values[left][right], bins=VIOLIN_BINS, min=-1, max=1)
            density = F.conv1d(density.view(1, 1, -1), VIOLIN_SMOOTHING, padding=2).flatten()
            width = 0.43 * density / density.max().clamp_min(1)
            x += (right - width).tolist() + (right + width).flip(0).tolist() + [None]
            y += (left + 0.42 * centers).tolist() + (left + 0.42 * centers).flip(0).tolist() + [None]
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines', fill='toself', fillcolor='rgba(255,255,255,0.88)',
        line={'color': 'rgba(31,41,55,0.55)', 'width': 0.6}, hoverinfo='skip', showlegend=False,
    ), row=1, col=column)

fig.update_xaxes(title_text='layer', dtick=1)
fig.update_yaxes(title_text='layer', dtick=1, autorange='reversed')

fig = make_subplots(
    rows=1, cols=2, subplot_titles=('Residual stream (layer 0 = embeddings)', 'Residual-write similarity'),
    horizontal_spacing=0.10,
)
add_similarity_heatmap_with_violins(fig, 'residual', 1)
add_similarity_heatmap_with_violins(fig, 'residual_write', 2)
fig.update_layout(
    title='Corresponding-token cosine similarity between layers (mini violin: height = cosine, width = density)',
    width=1_350, height=720,
)
fig.show()
